This notebook trains an ensemble of classifiers and regressors with a simple
logistic meta-model. The feature engineering is configurable so you can choose
which feature blocks each model trains on, while keeping the fold logic and
regression-output preprocessing in one place.

In [1]:
# Install only if needed in a fresh environment.
# !pip install lightgbm xgboost catboost scikit-learn scipy -q

import time
import warnings

#import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier#, CatBoostRegressor
#from scipy.special import expit
#from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import (
    #ElasticNet,
    LogisticRegression,
    #LogisticRegressionCV,
    #RidgeCV,
    #RidgeClassifierCV,
)
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import KFold, StratifiedKFold
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import QuantileTransformer, StandardScaler
# from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
print("Imports ready")



Imports ready


In [2]:
#Seed congifurations 
SEED_CONFIGS = [
    {"cv": 0, "lgbm": 42,  "cat": 42},
    {"cv": 1, "lgbm": 137, "cat": 202},
    {"cv": 2, "lgbm": 579, "cat": 381},
    {"cv": 3, "lgbm": 926, "cat": 781},
    {"cv": 4, "lgbm": 536, "cat": 303},
]
#Random seed for everything else
RANDOM_STATE = 42
# NUmber of CV folds
N_SPLITS = 8

print(f"This will train {N_SPLITS} fold CV models for three sets of seeds")

This will train 8 fold CV models for three sets of seeds


In [3]:

# Paths and run configuration

# Regression-output preprocessing hooks.
# Use quantile_regression_outputs to map raw regression outputs onto 0..1.
# def quantile_regression_outputs(oof_raw, test_raw, random_state=RANDOM_STATE):
#     qt = QuantileTransformer(
#         output_distribution="uniform",
#         n_quantiles=min(1000, len(oof_raw)),
#         random_state=random_state,
#     )
#     oof_scaled = qt.fit_transform(np.asarray(oof_raw).reshape(-1, 1)).ravel()
#     test_scaled = qt.transform(np.asarray(test_raw).reshape(-1, 1)).ravel()
#     return oof_scaled, test_scaled


# Edit these paths for your environment.
train_path = "/kaggle/input/datasets/akshayyelleshpur/qrt-trust-or-short/train.csv"
test_path = "/kaggle/input/datasets/akshayyelleshpur/qrt-trust-or-short/test.csv"

# Choose the models to train.
# You can add or remove entries without touching the training code.
RUN_MODELS = [
    "lgbm_clf",
    #"lgbm_reg",
    #"elasticnet",
    #"elasticnet_reg",
    # "ridge_clf",
    # "ridge_reg",
     #"xgb_clf",
    # "xgb_reg",
     "cat_clf",
    # "cat_reg",
]

# For regressors, choose either:
#   - quantile_regression_outputs: map outputs to 0..1 before stacking
#   - None: keep raw regression outputs and feed them directly to the meta-model
#DEFAULT_REG_POSTPROCESSOR = quantile_regression_outputs


# Feature selection controls.
# Edit FEATURE_BLOCKS_TO_USE to choose which engineered feature blocks are used.
# The same selected feature set is used for every model by default.
FEATURE_BLOCKS_TO_USE = [
    "return_stats",
    "volume_stats",
    "cross_features",
    "allocation_stats",
    "mdt",
    "group_ohe",
    "allocation_ohe",
]

# Optional: override the global feature set for a specific model key.
# Example:
# MODEL_FEATURE_BLOCKS = {
#     "ridge_clf": ["return_stats", "allocation_stats", "allocation_ohe"],
# }
MODEL_FEATURE_BLOCKS = {"ridge_clf": [],"lgbm_clf":["allocation_ohe","allocation_stats"],"xgb_clf":[],"cat_clf":["missing_indicators","return_stats","volume_stats","cross_features","mdt","group_ohe"]}



print("Config loaded")

Config loaded


In [4]:
# GPU detection
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    USE_GPU = result.returncode == 0
    if USE_GPU:
        print(f"GPU detected: {result.stdout.strip()}")
    else:
        print("No GPU detected; using CPU")
except FileNotFoundError:
    USE_GPU = False
    print("nvidia-smi not found; using CPU")


GPU detected: Tesla P100-PCIE-16GB


In [12]:
# Feature engineering

RET_COLS = [f"RET_{i}" for i in range(1, 21)]
VOL_COLS = [f"SIGNED_VOLUME_{i}" for i in range(2, 21)]  # drop SIGNED_VOLUME_1


def vec_stats(M, prefix):
    feats = {}
    N, T = M.shape

    feats[f"{prefix}_mean"] = np.nanmean(M, axis=1)
    feats[f"{prefix}_std"] = np.nanstd(M, axis=1)
    feats[f"{prefix}_min"] = np.nanmin(M, axis=1)
    feats[f"{prefix}_max"] = np.nanmax(M, axis=1)
    feats[f"{prefix}_range"] = feats[f"{prefix}_max"] - feats[f"{prefix}_min"]

    feats[f"{prefix}_sharpe"] = np.where(
        feats[f"{prefix}_std"] > 1e-9,
        feats[f"{prefix}_mean"] / feats[f"{prefix}_std"],
        0.0,
    )

    feats[f"{prefix}_pct_pos"] = np.nanmean(M > 0, axis=1)

    feats[f"{prefix}_last1"] = M[:, -1]
    feats[f"{prefix}_last3"] = np.nanmean(M[:, -3:], axis=1)
    feats[f"{prefix}_last5"] = np.nanmean(M[:, -5:], axis=1)
    feats[f"{prefix}_last10"] = np.nanmean(M[:, -10:], axis=1)
    feats[f"{prefix}_first5"] = np.nanmean(M[:, :5], axis=1)

    feats[f"{prefix}_mom_5_20"] = feats[f"{prefix}_last5"] - feats[f"{prefix}_first5"]
    feats[f"{prefix}_mom_3_10"] = np.nanmean(M[:, -3:], axis=1) - np.nanmean(M[:, 7:17], axis=1)

    feats[f"{prefix}_vol5"] = np.nanstd(M[:, -5:], axis=1)
    feats[f"{prefix}_vol10"] = np.nanstd(M[:, -10:], axis=1)
    feats[f"{prefix}_vol_ratio"] = np.where(
        feats[f"{prefix}_vol10"] > 1e-9,
        feats[f"{prefix}_vol5"] / feats[f"{prefix}_vol10"],
        1.0,
    )

    x = np.arange(T, dtype=np.float32)
    xm = x.mean()
    xv = ((x - xm) ** 2).sum()
    feats[f"{prefix}_trend"] = np.where(
        xv > 0,
        ((M - np.nanmean(M, axis=1, keepdims=True)) * (x - xm)).sum(axis=1) / xv,
        0.0,
    )

    mu = np.nanmean(M, axis=1, keepdims=True)
    std = np.nanstd(M, axis=1, keepdims=True) + 1e-9
    z = (M - mu) / std
    feats[f"{prefix}_skew"] = np.nanmean(z ** 3, axis=1)
    feats[f"{prefix}_kurt"] = np.nanmean(z ** 4, axis=1) - 3.0

    M0 = M[:, :-1] - np.nanmean(M[:, :-1], axis=1, keepdims=True)
    M1 = M[:, 1:] - np.nanmean(M[:, 1:], axis=1, keepdims=True)
    num = np.nansum(M0 * M1, axis=1)
    den = np.sqrt(np.nansum(M0**2, axis=1) * np.nansum(M1**2, axis=1))
    feats[f"{prefix}_autocorr1"] = np.where(den > 1e-9, num / den, 0.0)

    s5 = np.sign(M[:, -5:])
    feats[f"{prefix}_streak5"] = (
        (s5 > 0).all(axis=1).astype(np.float32)
        - (s5 < 0).all(axis=1).astype(np.float32)
    )

    return feats

def compute_alloc_stats(df, target_col='TARGET'):
    # # 1. Isolate the 20 daily return columns
    # ret_cols = [f'RET_{i}' for i in range(1, 21)]
    
    # # 2. Calculate row-by-row moments horizontally across the 20 days
    # df['_temp_row_mean'] = df[ret_cols].mean(axis=1)
    # df['_temp_row_vol']  = df[ret_cols].std(axis=1)
    # df['_temp_row_skew'] = df[ret_cols].skew(axis=1)
    # df['_temp_row_kurt'] = df[ret_cols].kurt(axis=1)
    # df['_temp_row_sharpe'] = df['_temp_row_mean'] / (df['_temp_row_vol'] + 1e-8)
    
    # # 3. Create the indicator for the positive sign binary target
    df['_is_positive'] = (df[target_col] > 0).astype(float)
    
    # 4. Single-pass aggregation by ALLOCATION (Using function pointers for kurtosis)
    grouped = df.groupby("ALLOCATION").agg(
        # Target Moments
        alloc_win_rate=('_is_positive', 'mean'),
        alloc_mean_ret=(target_col, 'mean'),
        alloc_vol=(target_col, 'std'),
        alloc_skew=(target_col, 'skew'),     
        alloc_kurt=(target_col, pd.Series.kurt))  # FIXED: Pass the actual function instead of the string 'kurt'
        
        # Aggregated 20-Day Feature Moments
    #     alloc_avg_row_mean=('_temp_row_mean', 'mean'),
    #     alloc_avg_row_vol=('_temp_row_vol', 'mean'),
    #     alloc_avg_row_skew=('_temp_row_skew', 'mean'),
    #     alloc_avg_row_kurt=('_temp_row_kurt', 'mean'), # FIXED: '_temp_row_kurt' is already aggregated horizontally, so we take the 'mean' here
    #     alloc_avg_row_sharpe=('_temp_row_sharpe', 'mean')
    # )
    
    # 5. Vectorized calculation for target Sharpe ratio
    grouped['alloc_sharpe'] = grouped['alloc_mean_ret'] / (grouped['alloc_vol'] + 1e-8)
    
    # 6. Clean up all temporary columns from source dataframe
    # temp_cols = ['_temp_row_mean', '_temp_row_vol', '_temp_row_skew', 
    #              '_temp_row_kurt', '_temp_row_sharpe', '_is_positive']
    # df.drop(columns=temp_cols, inplace=True, errors='ignore')
    
    # 7. Convert columns into a dictionary of pandas Series
    return {
        "alloc_win_rate": grouped["alloc_win_rate"],
        "alloc_mean_ret": grouped["alloc_mean_ret"],
        "alloc_vol": grouped["alloc_vol"],
        "alloc_skew": grouped["alloc_skew"],
        "alloc_kurt": grouped["alloc_kurt"],
        "alloc_sharpe": grouped["alloc_sharpe"],
        
        # "alloc_avg_row_mean": grouped["alloc_avg_row_mean"],
        # "alloc_avg_row_vol": grouped["alloc_avg_row_vol"],
        # "alloc_avg_row_skew": grouped["alloc_avg_row_skew"],
        # "alloc_avg_row_kurt": grouped["alloc_avg_row_kurt"],
        # "alloc_avg_row_sharpe": grouped["alloc_avg_row_sharpe"]
    }

def engineer_features(df, ohe_columns=None, alloc_stats=None):
    t0 = time.time()
    R = df[RET_COLS].values.astype(np.float32)  # (N, 20)
    
    feats = {}
    feats["vol1_missing"] = df["SIGNED_VOLUME_1"].isna().astype(np.float32).values
    
    V = df[VOL_COLS].values.astype(np.float32)  # (N, 19) — cols 2-20
    feats.update(vec_stats(R, "ret"))
    
    cumret = np.cumprod(1 + np.nan_to_num(R), axis=1) - 1
    running_max = np.maximum.accumulate(cumret, axis=1)
    dd = cumret - running_max
    feats["ret_max_dd"] = dd.min(axis=1)
    feats["ret_calmar"] = feats["ret_mean"] / (np.abs(feats["ret_max_dd"]) + 1e-9)
    feats.update(vec_stats(V, "vol"))
    
    # Align R to V's time axis: drop RET_1 (index 0) to match SIGNED_VOLUME_2..20
    R_aligned = R[:, 1:]  # (N, 19) — RET_2 through RET_20
    Rm = R_aligned - np.nanmean(R_aligned, axis=1, keepdims=True)
    Vm = V - np.nanmean(V, axis=1, keepdims=True)
    num = np.nansum(Rm * Vm, axis=1)
    den = np.sqrt(np.nansum(Rm**2, axis=1) * np.nansum(Vm**2, axis=1))
    feats["ret_vol_corr"] = np.where(den > 1e-9, num / den, 0.0)
    feats["vol_on_up"] = np.nanmean(np.where(R_aligned > 0, V, np.nan), axis=1)
    feats["vol_on_down"] = np.nanmean(np.where(R_aligned < 0, V, np.nan), axis=1)
    feats["vol_bias"] = feats["vol_on_up"] - feats["vol_on_down"]
    feats["mdt"] = df["MEDIAN_DAILY_TURNOVER"].values.astype(np.float32)
    
    if alloc_stats is not None:
        for stat_name, stat_series in alloc_stats.items():
            feats[stat_name] = df["ALLOCATION"].map(stat_series).values.astype(np.float32)
    feat_df = pd.DataFrame(feats, index=df.index)
    for col in ["GROUP", "ALLOCATION"]:
        if col not in df.columns:
            continue
        dummies = pd.get_dummies(df[col].astype(str), prefix=col, dtype=np.float32)
        if ohe_columns is not None and col in ohe_columns:
            dummies = dummies.reindex(columns=ohe_columns[col], fill_value=0.0)
        feat_df = pd.concat([feat_df, dummies], axis=1)
    print(f"  Feature engineering done in {time.time() - t0:.1f}s ({feat_df.shape[1]} features)")
    return feat_df

FEATURE_BLOCK_MAP = {
    "return_stats": lambda cols: [
        c for c in cols if c.startswith("ret_") or c in {"ret_max_dd", "ret_calmar"}
    ],
    "volume_stats": lambda cols: [
        c for c in cols if c.startswith("vol_")
    ],
    "cross_features": lambda cols: [
        c for c in cols if c in {"ret_vol_corr", "vol_on_up", "vol_on_down", "vol_bias"}
    ],
    "allocation_stats": lambda cols: [
        c for c in cols if c.startswith("alloc_")
    ],
    "mdt": lambda cols: ["mdt"] if "mdt" in cols else [],
    "group_ohe": lambda cols: [c for c in cols if c.startswith("GROUP_")],
    "allocation_ohe": lambda cols: [c for c in cols if c.startswith("ALLOCATION_")],
    "missing_indicators": lambda cols: [c for c in cols if c.endswith("_missing")],
}


def select_feature_columns(all_columns, feature_blocks):
    """Return a stable feature subset given an ordered list of feature blocks."""
    if feature_blocks is None:
        return list(all_columns)

    selected = []
    seen = set()

    for block in feature_blocks:
        if block == "all":
            return list(all_columns)
        if block not in FEATURE_BLOCK_MAP:
            raise ValueError(
                f"Unknown feature block '{block}'. Available blocks: {sorted(FEATURE_BLOCK_MAP)}"
            )
        cols = FEATURE_BLOCK_MAP[block](all_columns)
        for col in cols:
            if col in all_columns and col not in seen:
                selected.append(col)
                seen.add(col)

    return selected

print("Feature engineering ready")


Feature engineering ready


In [13]:
# Model factories

def get_lgbm_classifier():
    params = dict(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=5,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight=None,
        random_state=lgbm_seed,
        n_jobs=-1,
        verbose=-1,
    )
    if USE_GPU:
        params["device"] = "gpu"
    return lgb.LGBMClassifier(**params)


# def get_lgbm_regressor():
#     params = dict(
#         n_estimators=5000,
#         learning_rate=0.03,
#         num_leaves=31,
#         max_depth=5,
#         min_child_samples=50,
#         subsample=0.8,
#         subsample_freq=1,
#         colsample_bytree=0.8,
#         reg_alpha=0.1,
#         reg_lambda=1.0,
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#         verbose=-1,
#     )
#     if USE_GPU:
#         params["device"] = "gpu"
#     return lgb.LGBMRegressor(**params)


# def get_elasticnet_clf():
#     return Pipeline(
#         [
#             ("scaler", StandardScaler()),
#             (
#                 "clf",
#                 LogisticRegression(
#                     penalty="elasticnet",
#                     solver="saga",
#                     l1_ratio=0.5,
#                     C=0.1,
#                     class_weight="balanced",
#                     max_iter=200,
#                     tol=1e-2,
#                     random_state=RANDOM_STATE,
#                     n_jobs=-1,
#                 ),
#             ),
#         ]
#     )


# def get_elasticnet_regressor():
#     return Pipeline(
#         [
#             ("scaler", StandardScaler()),
#             (
#                 "reg",
#                 ElasticNet(
#                     alpha=0.01,
#                     l1_ratio=0.5,
#                     max_iter=2000,
#                     random_state=RANDOM_STATE,
#                 ),
#             ),
#         ]
#     )


# def get_ridge_classifier():
#     return RidgeClassifierCV(alphas=np.logspace(-3, 3, 13))


# def get_ridge_regressor():
#     return RidgeCV(alphas=np.logspace(-3, 3, 13))


# def get_xgb_classifier():
#     params = dict(
#         n_estimators=5000,
#         learning_rate=0.03,
#         max_depth=4,
#         min_child_weight=10,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_alpha=0.1,
#         reg_lambda=1.0,
#         objective="binary:logistic",
#         eval_metric="logloss",
#         early_stopping_rounds=50,
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#         tree_method="hist",
#     )
#     if USE_GPU:
#         params["device"] = "cuda"
#     return XGBClassifier(**params)


# def get_xgb_regressor():
#     params = dict(
#         n_estimators=5000,
#         learning_rate=0.03,
#         max_depth=4,
#         min_child_weight=10,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_alpha=0.1,
#         reg_lambda=1.0,
#         objective="reg:squarederror",
#         eval_metric="rmse",
#         early_stopping_rounds=50,
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#         tree_method="hist",
#     )
#     if USE_GPU:
#         params["device"] = "cuda"
#     return XGBRegressor(**params)


def get_cat_classifier():
    params = dict(
        iterations=5000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        loss_function="Logloss",
        eval_metric="Logloss",
        early_stopping_rounds=50,
        random_seed=cat_seed,
        verbose=False,
    )
    if USE_GPU:
        params["task_type"] = "GPU"
    return CatBoostClassifier(**params)


# def get_cat_regressor():
#     params = dict(
#         iterations=5000,
#         learning_rate=0.03,
#         depth=6,
#         l2_leaf_reg=5.0,
#         loss_function="RMSE",
#         eval_metric="RMSE",
#         early_stopping_rounds=50,
#         random_seed=RANDOM_STATE,
#         verbose=False,
#     )
#     if USE_GPU:
#         params["task_type"] = "GPU"
#     return CatBoostRegressor(**params)

print("Model factories ready")


Model factories ready


In [14]:
MODEL_SPECS = {
    "lgbm_clf": {
        "display": "LightGBM Classifier",
        "kind": "clf",
        "matrix": "trees",
        "factory": get_lgbm_classifier,
        "show_importance": True,
    },
    # "lgbm_reg": {
    #     "display": "LightGBM Regressor",
    #     "kind": "reg",
    #     "matrix": "trees",
    #     "factory": get_lgbm_regressor,
    #     "postprocess": DEFAULT_REG_POSTPROCESSOR,
    #     "show_importance": True,
    # },
    # "elasticnet": {
    #     "display": "ElasticNet Logistic Regression",
    #     "kind": "clf",
    #     "matrix": "linear",
    #     "factory": get_elasticnet_clf,
    #     "show_importance": False,
    # },
    # "elasticnet_reg": {
    #     "display": "ElasticNet Linear Regressor",
    #     "kind": "reg",
    #     "matrix": "linear",
    #     "factory": get_elasticnet_regressor,
    #     "postprocess": DEFAULT_REG_POSTPROCESSOR,
    #     "show_importance": False,
    # },
    # "ridge_clf": {
    #     "display": "Ridge Classifier",
    #     "kind": "clf",
    #     "matrix": "linear",
    #     "factory": get_ridge_classifier,
    #     "show_importance": False,
    # },
    # "ridge_reg": {
    #     "display": "Ridge Regressor",
    #     "kind": "reg",
    #     "matrix": "linear",
    #     "factory": get_ridge_regressor,
    #     "postprocess": DEFAULT_REG_POSTPROCESSOR,
    #     "show_importance": False,
    # },
    # "xgb_clf": {
    #     "display": "XGBoost Classifier",
    #     "kind": "clf",
    #     "matrix": "trees",
    #     "factory": get_xgb_classifier,
    #     "show_importance": True,
    # },
    # "xgb_reg": {
    #     "display": "XGBoost Regressor",
    #     "kind": "reg",
    #     "matrix": "trees",
    #     "factory": get_xgb_regressor,
    #     "postprocess": DEFAULT_REG_POSTPROCESSOR,
    #     "show_importance": True,
    # },
    "cat_clf": {
        "display": "CatBoost Classifier",
        "kind": "clf",
        "matrix": "trees",
        "factory": get_cat_classifier,
        "show_importance": False,
     },
    # "cat_reg": {
    #     "display": "CatBoost Regressor",
    #     "kind": "reg",
    #     "matrix": "trees",
    #     "factory": get_cat_regressor,
    #     "postprocess": DEFAULT_REG_POSTPROCESSOR,
    #     "show_importance": False,
    # },
}

print("Model specs loaded")

Model specs loaded


In [25]:
# CV and training helpers

def make_date_kfolds(df_raw, random_state, n_splits=8):
    dates = df_raw["TS"].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    pos_map = {idx: pos for pos, idx in enumerate(df_raw.index)}
    folds_pos = []
    for train_date_idx, val_date_idx in kf.split(dates):
        train_dates = dates[train_date_idx]
        val_dates = dates[val_date_idx]
        tr_idx = np.array([pos_map[i] for i in df_raw.index[df_raw["TS"].isin(train_dates)]])
        vl_idx = np.array([pos_map[i] for i in df_raw.index[df_raw["TS"].isin(val_dates)]])
        folds_pos.append((tr_idx, vl_idx))
    return folds_pos


def _build_fold_arrays(df_raw, tr_idx, vl_idx, ohe_columns, col_names, selected_cols, is_linear=False):
    """Build leak-free train/val numpy arrays for one fold."""
    #fold_alloc_stats = compute_alloc_stats(df_raw.iloc[tr_idx])
    
    X_tr_df = engineer_features(df_raw.iloc[tr_idx], ohe_columns=ohe_columns, alloc_stats=None)
    X_vl_df = engineer_features(df_raw.iloc[vl_idx], ohe_columns=ohe_columns, alloc_stats=None)
    
    X_tr_df = X_tr_df.reindex(columns=selected_cols, fill_value=0)
    X_vl_df = X_vl_df.reindex(columns=selected_cols, fill_value=0)
    
    X_tr = X_tr_df.values.astype(np.float32)
    X_vl = X_vl_df.values.astype(np.float32)
    
    # if is_linear:
    #     col_medians = np.nanmedian(X_tr, axis=0)
    #     nan_tr = np.isnan(X_tr)
    #     nan_vl = np.isnan(X_vl)
    #     X_tr[nan_tr] = np.take(col_medians, np.where(nan_tr)[1])
    #     X_vl[nan_vl] = np.take(col_medians, np.where(nan_vl)[1])
    return X_tr, X_vl



def _score_like_classifier(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    # if hasattr(model, "decision_function"):
    #     return expit(model.decision_function(X))
    return np.asarray(model.predict(X), dtype=float)


def train_oof_classifier(df_raw, y_clf, folds_pos, model_fn, name, ohe_columns, col_names, selected_cols, is_linear=False):
    oof = np.full(len(y_clf), np.nan)
    models = []
    for k, (tr_idx, vl_idx) in enumerate(folds_pos):
        t0 = time.time()
        X_tr, X_vl = _build_fold_arrays(df_raw, tr_idx, vl_idx, ohe_columns, col_names, selected_cols, is_linear)
        m = model_fn()
        if isinstance(m, lgb.LGBMClassifier):
            m.fit(
                X_tr, y_clf[tr_idx],
                eval_set=[(X_vl, y_clf[vl_idx])],
                eval_metric="binary_logloss",
                callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)],
            )
            pred = m.predict_proba(X_vl, num_iteration=getattr(m, "best_iteration_", None))[:, 1]
        # elif isinstance(m, XGBClassifier):
        #     m.fit(X_tr, y_clf[tr_idx], eval_set=[(X_vl, y_clf[vl_idx])], verbose=False)
        #     pred = _score_like_classifier(m, X_vl)
        elif isinstance(m, CatBoostClassifier):
            m.fit(X_tr, y_clf[tr_idx], eval_set=(X_vl, y_clf[vl_idx]), use_best_model=True, verbose=False)
            pred = _score_like_classifier(m, X_vl)
        else:
            m.fit(X_tr, y_clf[tr_idx])
            pred = _score_like_classifier(m, X_vl)
        oof[vl_idx] = pred
        models.append(m)
        acc = accuracy_score(y_clf[vl_idx], (pred > 0.5).astype(int))
        print(f"  [{name}] fold {k + 1}/{len(folds_pos)}  val_acc={acc:.4f}  ({time.time() - t0:.0f}s)")
    covered = ~np.isnan(oof)
    oof_acc = accuracy_score(y_clf[covered], (oof[covered] > 0.5).astype(int))
    print(f"  -> {name} OOF accuracy: {oof_acc:.4f}\n")
    return oof, models, oof_acc


# def train_oof_regressor(df_raw, y_reg, y_clf, folds_pos, model_fn, name, ohe_columns, col_names, selected_cols, is_linear=False):
#     oof = np.full(len(y_reg), np.nan)
#     models = []
#     for k, (tr_idx, vl_idx) in enumerate(folds_pos):
#         t0 = time.time()
#         X_tr, X_vl = _build_fold_arrays(df_raw, tr_idx, vl_idx, ohe_columns, col_names, selected_cols, is_linear)
#         m = model_fn()
#         if isinstance(m, lgb.LGBMRegressor):
#             m.fit(
#                 X_tr, y_reg[tr_idx],
#                 eval_set=[(X_vl, y_reg[vl_idx])],
#                 eval_metric="l2",
#                 callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)],
#             )
#             pred = m.predict(X_vl, num_iteration=getattr(m, "best_iteration_", None))
#         elif isinstance(m, XGBRegressor):
#             m.fit(X_tr, y_reg[tr_idx], eval_set=[(X_vl, y_reg[vl_idx])], verbose=False)
#             pred = m.predict(X_vl)
#         elif isinstance(m, CatBoostRegressor):
#             m.fit(X_tr, y_reg[tr_idx], eval_set=(X_vl, y_reg[vl_idx]), use_best_model=True, verbose=False)
#             pred = m.predict(X_vl)
#         else:
#             m.fit(X_tr, y_reg[tr_idx])
#             pred = m.predict(X_vl)
#         oof[vl_idx] = pred
#         models.append(m)
#         acc = accuracy_score(y_clf[vl_idx], (pred > 0).astype(int))
#         print(f"  [{name}] fold {k + 1}/{len(folds_pos)}  val_acc={acc:.4f}  ({time.time() - t0:.0f}s)")
#     covered = ~np.isnan(oof)
#     oof_acc = accuracy_score(y_clf[covered], (oof[covered] > 0).astype(int))
#     print(f"  -> {name} OOF accuracy (sign): {oof_acc:.4f}\n")
#     return oof, models, oof_acc


def predict_ensemble_classifier(models, X):
    return np.stack([_score_like_classifier(m, X) for m in models], axis=1).mean(axis=1)


# def predict_ensemble_regressor(models, X):
#     return np.stack([m.predict(X) for m in models], axis=1).mean(axis=1)



# Regression-output preprocessing helpers are defined earlier in the notebook so
# they can be selected from the configuration cell.


print("Training helpers ready")

Training helpers ready


In [26]:
# Load and prepare data

t_start = time.time()
print("Loading data...")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f"  Train shape: {train.shape}")
print(f"  Test shape : {test.shape}")

y_clf = (train["TARGET"] > 0).astype(int).values
y_reg = train["TARGET"].values.astype(np.float32)
global_train_positive_rate = y_clf.mean()
print(f"  Class balance: {y_clf.mean():.3f} positive")

#alloc_stats_full = compute_alloc_stats(train)
_schema_df = engineer_features(train, alloc_stats=None)
ohe_columns = {
    col: [c for c in _schema_df.columns if c.startswith(col + "_")]
    for col in ["GROUP","ALLOCATION"]
}
col_names = list(_schema_df.columns)
del _schema_df

selected_cols = select_feature_columns(col_names, FEATURE_BLOCKS_TO_USE)
#print(f"  Using {len(selected_cols)} of {len(col_names)} engineered features")
#print(f"  Feature blocks: {FEATURE_BLOCKS_TO_USE}")

#This is only necessary in case we need to build the input for regression models. 
#In this case, we cannot have NaNs and we need to impute the test dataset with medians from the train.
# train_feat_full_df = engineer_features(train, ohe_columns=ohe_columns, alloc_stats=alloc_stats_full)
# train_feat_full_df = train_feat_full_df.reindex(columns=col_names, fill_value=0)

test_feat_full_df = engineer_features(test, ohe_columns=ohe_columns, alloc_stats=None)
test_feat_full_df = test_feat_full_df.reindex(columns=col_names, fill_value=0)



Loading data...
  Train shape: (527073, 45)
  Test shape : (31870, 45)
  Class balance: 0.507 positive
  Feature engineering done in 6.2s (334 features)
  Feature engineering done in 0.3s (334 features)


In [27]:
all_oof = []
all_test = []
all_features = []

In [28]:
for seeds in SEED_CONFIGS:
    cv_seed = seeds["cv"]
    lgbm_seed = seeds["lgbm"]
    cat_seed = seeds["cat"]
    print(f"Running seed configuration {seeds}")

    
    folds_pos = make_date_kfolds(df_raw=train, n_splits=N_SPLITS, random_state=cv_seed)
    print(f"  {N_SPLITS}-fold date-based CV ready using seed {cv_seed}")
    print(f"  Avg train size: {np.mean([len(tr) for tr, _ in folds_pos]):.0f} rows")
    print(f"  Avg val size  : {np.mean([len(vl) for _, vl in folds_pos]):.0f} rows")

    oof_probas = {}
    test_probas = {}
    oof_accs = {}
    trained_models = {}
    feature_importance_rows = []
    
    for key in RUN_MODELS:
        spec = MODEL_SPECS[key]
        is_linear = spec["matrix"] == "linear"
        feature_blocks = MODEL_FEATURE_BLOCKS.get(key, FEATURE_BLOCKS_TO_USE)
        selected_cols_model = select_feature_columns(col_names, feature_blocks)
        print(f"Training {spec['display']} ...")
        print(f"  Feature blocks: {feature_blocks}")
        print(f"  Selected features: {len(selected_cols_model)}")
    
        X_test_model_df = test_feat_full_df.reindex(columns=selected_cols_model, fill_value=0)
        X_test_model = X_test_model_df.values.astype(np.float32)
        
        # if is_linear:
        #     X_train_model_df = train_feat_full_df.reindex(columns=selected_cols_model, fill_value=0)
        #     col_medians_global = np.nanmedian(X_train_model_df.values.astype(np.float32), axis=0)
        #     X_test_model = X_test_model.copy()
        #     nan_mask = np.isnan(X_test_model)
        #     X_test_model[nan_mask] = np.take(col_medians_global, np.where(nan_mask)[1])
    
        # if spec["kind"] == "clf":
        oof, models, acc = train_oof_classifier(
            train, y_clf, folds_pos, spec["factory"], spec["display"],
            ohe_columns, col_names, selected_cols_model, is_linear=is_linear,
        )
        test_pred = predict_ensemble_classifier(models, X_test_model)
        oof_pred = oof
        # else:
        #     oof, models, acc = train_oof_regressor(
        #         train, y_reg, y_clf, folds_pos, spec["factory"], spec["display"],
        #         ohe_columns, col_names, selected_cols_model, is_linear=is_linear,
        #     )
        #     test_raw = predict_ensemble_regressor(models, X_test_model)
        #     postprocess_fn = spec.get("postprocess")
        #     if callable(postprocess_fn):
        #         oof_pred, test_pred = postprocess_fn(oof, test_raw, random_state=RANDOM_STATE)
        #     else:
        #         oof_pred, test_pred = oof, test_raw
    
        oof_probas[key] = oof_pred
        test_probas[key] = test_pred
        #if key=="lgbm_clf":
        #     mask_test = test["ALLOCATION"].isin([14, 46])
        #     test_probas[key][mask_test] = global_train_positive_rate
        oof_accs[key] = acc
        trained_models[key] = models
    

        if len(models) > 0:

            # LightGBM
            if hasattr(models[0], "feature_importances_"):
                importances = np.mean(
                    [m.feature_importances_ for m in models],
                    axis=0
                )
        
            # CatBoost
            elif hasattr(models[0], "get_feature_importance"):
                importances = np.mean(
                    [m.get_feature_importance() for m in models],
                    axis=0
                )
        
            else:
                importances = None
        
            if importances is not None:
        
                for feat, imp in zip(selected_cols_model, importances):
                    feature_importance_rows.append({
                        "cv_seed": cv_seed,
                        "lgbm_seed": lgbm_seed,
                        "cat_seed": cat_seed,
                        "model": key,
                        "feature": feat,
                        "importance": float(imp),
                    })

    
    all_oof.append(oof_probas)
    all_test.append(test_probas)
    
    print("=" * 45)
    print(f"{'Model':<20} {'OOF Accuracy':>12}")
    print("=" * 45)
    for name, acc in oof_accs.items():
        print(f"  {name:<18} {acc:>12.4f}")
    print("=" * 45)


Running seed configuration {'cv': 0, 'lgbm': 42, 'cat': 42}
  8-fold date-based CV ready using seed 0
  Avg train size: 461189 rows
  Avg val size  : 65884 rows
Training LightGBM Classifier ...
  Feature blocks: ['allocation_ohe', 'allocation_stats']
  Selected features: 278
  Feature engineering done in 5.6s (334 features)
  Feature engineering done in 0.7s (334 features)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[424]	valid_0's binary_logloss: 0.691562
  [LightGBM Classifier] fold 1/8  val_acc=0.5153  (14s)
  Feature engineering done in 5.6s (334 features)
  Feature engineering done in 0.7s (334 features)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[580]	valid_0's binary_logloss: 0.691771
  [LightGBM Classifier] fold 2/8  val_acc=0.5151  (16s)
  Feature engineering done in 5.6s (334 features)
  Feature engineering done in 0.6s (334 features)
Training until validation scores don't

In [30]:
oof_avg = {
    model_name: np.mean(
        [run_oof[model_name] for run_oof in all_oof],
        axis=0
    )
    for model_name in RUN_MODELS
}

test_avg = {
    model_name: np.mean(
        [run_test[model_name] for run_test in all_test],
        axis=0
    )
    for model_name in RUN_MODELS
}

In [31]:
if len(all_oof) >= 1:
    print("Building stacked ensemble from all seed runs...")

    # Collect one column per model per seed run
    train_cols = {}
    test_cols = {}

    for run_idx, run_oof in enumerate(all_oof):
        for model_name in RUN_MODELS:
            train_cols[f"{model_name}_run{run_idx}"] = run_oof[model_name]

    for run_idx, run_test in enumerate(all_test):
        for model_name in RUN_MODELS:
            test_cols[f"{model_name}_run{run_idx}"] = run_test[model_name]

    X_oof = pd.DataFrame(train_cols)
    X_test_stack = pd.DataFrame(test_cols)

    X_oof["mean_pred"] = X_oof.mean(axis=1)
    X_oof["std_pred"] = X_oof.std(axis=1)
    X_oof["max_pred"] = X_oof.max(axis=1)
    X_oof["min_pred"] = X_oof.min(axis=1)

    X_test_stack["mean_pred"] = X_test_stack.mean(axis=1)
    X_test_stack["std_pred"] = X_test_stack.std(axis=1)
    X_test_stack["max_pred"] = X_test_stack.max(axis=1)
    X_test_stack["min_pred"] = X_test_stack.min(axis=1)


    # Drop rows with any missing stacked features
    mask = ~X_oof.isna().any(axis=1)

    X_meta = X_oof.loc[mask].values
    y_meta = y_clf[mask].values if hasattr(y_clf, "values") else y_clf[mask]

    print(f"Stack matrix shape: {X_meta.shape}")
    print(f"{'Base feature':<25} {'OOF Acc':>8}")
    print("-" * 35)
    for n in X_oof.columns:
        acc = accuracy_score(y_meta, (X_oof.loc[mask, n] > 0.5).astype(int))
        print(f"{n:<25} {acc:>8.4f}")
    print()

    # Honest OOF for the meta-model
    meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_stacked = np.zeros(len(y_meta), dtype=float)

    for tr_idx, va_idx in meta_cv.split(X_meta, y_meta):
        meta = LogisticRegression(
            C=1.0,
            max_iter=5000,
            solver="lbfgs",
            penalty="l2",
        )
        meta.fit(X_meta[tr_idx], y_meta[tr_idx])
        oof_stacked[va_idx] = meta.predict_proba(X_meta[va_idx])[:, 1]

    # Fit final meta-model on all stacked features
    final_meta = LogisticRegression(
        C=1.0,
        max_iter=5000,
        solver="lbfgs",
        penalty="l2",
    )
    final_meta.fit(X_meta, y_meta)

    final_proba = final_meta.predict_proba(X_test_stack.values)[:, 1]

    acc = accuracy_score(y_meta, (oof_stacked > 0.5).astype(int))
    ll = log_loss(y_meta, oof_stacked)

    print("Meta model: LogisticRegression")
    print(f"  Honest OOF accuracy: {acc:.4f}")
    print(f"  Honest OOF log loss: {ll:.4f}")
    print("  Coefficients:")
    for n, c in zip(X_oof.columns, final_meta.coef_[0]):
        print(f"    {n:<25} {c:>8.4f}")
    print()

Building stacked ensemble from all seed runs...
Stack matrix shape: (527073, 14)
Base feature               OOF Acc
-----------------------------------
lgbm_clf_run0               0.5131
cat_clf_run0                0.5156
lgbm_clf_run1               0.5133
cat_clf_run1                0.5155
lgbm_clf_run2               0.5128
cat_clf_run2                0.5151
lgbm_clf_run3               0.5122
cat_clf_run3                0.5156
lgbm_clf_run4               0.5128
cat_clf_run4                0.5165
mean_pred                   0.5174
std_pred                    0.4928
max_pred                    0.5108
min_pred                    0.4928

Meta model: LogisticRegression
  Honest OOF accuracy: 0.5179
  Honest OOF log loss: 0.6916
  Coefficients:
    lgbm_clf_run0               0.5319
    cat_clf_run0                0.4284
    lgbm_clf_run1               0.6132
    cat_clf_run1                0.6418
    lgbm_clf_run2               0.4872
    cat_clf_run2                0.4695
    lgbm_clf_run

In [14]:
# Save submission

predictions = (final_proba > 0.5).astype(int)
submission = test[["ROW_ID"]].copy()
submission["prediction"] = predictions
submission.to_csv("16-06:01.csv", index=False)

elapsed = time.time() - t_start
print(f"Total time: {elapsed / 60:.1f} min")
print(f"Predicted positive rate: {predictions.mean():.3f}")
print("submission_no_features_no_tuning.csv saved")


Total time: 24.5 min
Predicted positive rate: 0.605
submission_no_features_no_tuning.csv saved


In [32]:
#Generate and save files for analysis
train_diag= pd.DataFrame({
    "row_id": np.arange(len(train)),
    "target": y_clf,

    # useful original metadata
    "TS": train["TS"].values,
    "ALLOCATION": train["ALLOCATION"].values,

    # averaged OOF predictions
    "oof_lgbm_mean": oof_avg["lgbm_clf"],
    "oof_cat_mean": oof_avg["cat_clf"],
    "oof_stack": oof_stacked,   # if you compute OOF stack predictions

    # hard predictions at your chosen threshold
    "pred_lgbm": (oof_avg["lgbm_clf"] >= 0.5).astype(int),
    "pred_cat": (oof_avg["cat_clf"] >= 0.5).astype(int),
    "pred_stack": (oof_stacked >= 0.5).astype(int),

    # confidence
    "confidence": np.abs(oof_stacked - 0.5),
    "margin_from_threshold": oof_stacked - 0.5,
})

test_diag = pd.DataFrame({
    "row_id": np.arange(len(test)),
    "TS": test["TS"].values,
    "ALLOCATION": test["ALLOCATION"].values,

    "lgbm_mean": test_avg["lgbm_clf"],
    "cat_mean": test_avg["cat_clf"],
    "final_proba": final_proba,

    "prediction": (final_proba >= 0.5).astype(int),
    "confidence": np.abs(final_proba - 0.5),
    "margin_from_threshold": final_proba - 0.5,
})

feature_importance_df = pd.DataFrame(feature_importance_rows)
feature_importance_summary = (
    feature_importance_df
    .groupby(["model", "feature"])["importance"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["model", "mean"], ascending=[True, False])
)

feature_importance_df.to_parquet(
    "feature_importance_raw.parquet",
    index=False
)

feature_importance_summary.to_parquet(
    "feature_importance_summary.parquet",
    index=False
)
test_diag.to_parquet("test_prediction_diagnostics.parquet", index=False)
train_diag.to_parquet("train_prediction_diagnostics.parquet", index=False)

In [ ]:
from scipy.stats import spearmanr, pearsonr


def _as_1d(x):
    return np.asarray(x).ravel()


def model_to_sign_pred(preds, kind):
    """
    Convert model outputs to sign predictions.

    kind:
        - "clf_prob"      : classifier probability in [0, 1]
        - "reg_raw"       : raw regression output
        - "reg_quantile"  : quantile-transformed regression output in [0, 1]
    """
    preds = _as_1d(preds)

    if kind in {"clf_prob", "reg_quantile"}:
        return (preds >= 0.5).astype(int)
    elif kind == "reg_raw":
        return (preds >= 0).astype(int)
    else:
        raise ValueError(f"Unknown kind: {kind}")


def model_to_score(preds):
    """
    Return a numeric score for correlation calculations.
    Works for both raw regression outputs and probability-like outputs.
    """
    return _as_1d(preds)


def sign_accuracy(y_true, preds, kind):
    y_true = (_as_1d(y_true) > 0).astype(int)
    y_pred = model_to_sign_pred(preds, kind)
    return (y_true == y_pred).mean()


def build_prediction_frame(pred_dict, kind_dict):
    """
    pred_dict: {model_name: oof_predictions}
    kind_dict: {model_name: "clf_prob" | "reg_raw" | "reg_quantile"}
    """
    df = pd.DataFrame({
        name: model_to_score(preds)
        for name, preds in pred_dict.items()
    })
    return df


def build_correctness_frame(y_true, pred_dict, kind_dict):
    y_true_bin = (_as_1d(y_true) > 0).astype(int)

    df = pd.DataFrame({
        name: (model_to_sign_pred(preds, kind) == y_true_bin).astype(int)
        for name, preds, kind in [
            (name, pred_dict[name], kind_dict[name])
            for name in pred_dict.keys()
        ]
    })
    return df


def corr_matrices(pred_dict, kind_dict):
    """
    Returns:
        - Pearson correlation matrix of model outputs
        - Spearman correlation matrix of model outputs
        - Pearson correlation matrix of correctness indicators
    """
    pred_df = build_prediction_frame(pred_dict, kind_dict)
    corr_pearson = pred_df.corr(method="pearson")
    corr_spearman = pred_df.corr(method="spearman")

    return corr_pearson, corr_spearman


def correctness_corr_matrix(y_true, pred_dict, kind_dict):
    """
    Correlation of correctness vectors:
    1 = model got the sign right, 0 = wrong
    """
    correct_df = build_correctness_frame(y_true, pred_dict, kind_dict)
    return correct_df.corr(method="pearson")


def summary_table(y_true, pred_dict, kind_dict):
    rows = []
    for name in pred_dict:
        rows.append({
            "model": name,
            "sign_accuracy": sign_accuracy(y_true, pred_dict[name], kind_dict[name]),
        })
    return pd.DataFrame(rows).sort_values("sign_accuracy", ascending=False).reset_index(drop=True)

In [ ]:
def error_type(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)

    out = np.full(len(y), "TN", dtype=object)
    out[(y == 1) & (pred == 1)] = "TP"
    out[(y == 0) & (pred == 1)] = "FP"
    out[(y == 1) & (pred == 0)] = "FN"

    return out

train_diag["error_type"] = error_type(
    train_diag["target"].values,
    train_diag["oof_stack"].values,
    threshold=0.5
)

In [ ]:
pred_dict = {k: v for k, v in oof_probas.items()}

kind_dict = {}
for k in pred_dict:
    spec = MODEL_SPECS[k]
    if spec["kind"] == "clf":
        kind_dict[k] = "clf_prob"
    else:
        kind_dict[k] = "reg_quantile" if callable(spec.get("postprocess")) else "reg_raw"

print(summary_table(y_clf, pred_dict, kind_dict))

pearson_corr, spearman_corr = corr_matrices(pred_dict, kind_dict)
correct_corr = correctness_corr_matrix(y_clf, pred_dict, kind_dict)

display(pearson_corr)
display(spearman_corr)
display(correct_corr)